# Default notebook

This default notebook is executed using a Lakeflow job as defined in resources/sample_job.job.yml.

It also runs locally through Databricks Connect. Two things differ off-platform:

* `spark` and `dbutils` are injected into the namespace by the Databricks runtime, but not
  locally -- hence the explicit import below. The import is a no-op on Databricks, where it
  returns the runtime's own objects.
* Widget values arrive as job parameters when run as a task. Locally nothing supplies them, so
  `dbutils.widgets.get` raises `KeyError`. Declaring the widgets with a default fixes that, and
  is harmless on Databricks: an existing widget keeps the value the job passed it.


In [3]:
# `spark` and `dbutils` are provided automatically on Databricks. This import makes the same
# names available locally via Databricks Connect, and returns the runtime objects on Databricks.
from databricks.sdk.runtime import dbutils, spark

# Declare the parameters this notebook takes. resources/sample_job.job.yml passes `catalog` and
# `schema` as job parameters, which override these defaults on a job run.
dbutils.widgets.text("catalog", "workspace", "Catalog")
dbutils.widgets.text("schema", "default", "Schema")

# Set default catalog and schema
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
spark.sql(f"USE CATALOG `{catalog}`")
spark.sql(f"USE SCHEMA `{schema}`")

print(spark.sql("SELECT current_catalog(), current_schema()").collect()[0])


Row(current_catalog()='workspace', current_schema()='default')


In [0]:
import sys

sys.path.append("../src")
from smart_claims_dev import taxis

taxis.find_all_taxis().show(10)